# Task 072: ghost_imaging — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Computational ghost imaging using compressed sensing

**Paper**: See repository documentation
**Repository**: https://github.com/jovenitti/GhostImaging

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 50.18 dB ← 🔧 修复前: 14.11 dB
- **SSIM**: 0.998
- **Evaluation**: Compressive ghost imaging via FISTA (CC=0.887)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Ghost Imaging — Compressed-Sensing Single-Pixel Reconstruction
================================================================
Task #69: Reconstruct an image from single-pixel bucket detector
          measurements using compressive ghost imaging.

Inverse Problem:
    Given M bucket measurements b_i = <φ_i, x> + n_i, where φ_i are
    random speckle illumination patterns and x is the unknown image,
    recover x from M < N measurements (compressed sensing).

Forward Model:
    b = Φ x + n
    Φ is (M × N) measurement matrix (random speckle patterns),
    x is (N × 1) vectorised image, b is (M × 1) bucket signals.

Inverse Solver:
    1) Traditional correlation ghost imaging: x̂ = Σ (b_i - <b>) · φ_i
    2) ISTA (Iterative Shrinkage-Thresholding Algorithm) for CS
    3) FISTA (Fast ISTA) with TV or wavelet sparsity

Repo: https://github.com/cbasedlf/ghost_imaging_demo
Paper: Shapiro (2008), Phys. Rev. A; Katz et al. (2009), APL.

Usage: /data/yjh/spectro_env/bin/python ghost_imaging_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`correlation_gi`, `dct2d_basis_transform`, `soft_threshold`, `tv_denoise`

In [ ]:
# ─── Inverse Solver: Correlation Ghost Imaging ────────────────────
def correlation_gi(Phi, b):
    """
    Traditional correlation ghost imaging:
    x̂ = (1/M) Σ_i (b_i - <b>) · (φ_i - <φ>)
    """
    M = len(b)
    b_mean = b.mean()
    Phi_mean = Phi.mean(axis=0)
    x_rec = np.zeros(Phi.shape[1])
    for i in range(M):
        x_rec += (b[i] - b_mean) * (Phi[i] - Phi_mean)
    x_rec /= M
    return x_rec

# ─── Inverse Solver: ISTA ─────────────────────────────────────────
def dct2d_basis_transform(x, size, inverse=False):
    """2D DCT sparsifying transform."""
    X = x.reshape(size, size)
    if inverse:
        return idct(idct(X, axis=0, norm='ortho'), axis=1, norm='ortho').ravel()
    else:
        return dct(dct(X, axis=0, norm='ortho'), axis=1, norm='ortho').ravel()

def soft_threshold(x, tau):
    """Soft thresholding (proximal operator for L1)."""
    return np.sign(x) * np.maximum(np.abs(x) - tau, 0)

# ─── Total Variation Denoising ─────────────────────────────────────
def tv_denoise(img, weight=0.1, n_iter=50):
    """Simple isotropic TV denoising (Chambolle's projection)."""
    u = img.copy()
    px = np.zeros_like(u)
    py = np.zeros_like(u)
    tau = 0.25

    for _ in range(n_iter):
        # Gradient of u
        gx = np.diff(u, axis=0, append=u[-1:, :])
        gy = np.diff(u, axis=1, append=u[:, -1:])

        # Update dual
        px_new = px + tau * gx
        py_new = py + tau * gy
        norm = np.sqrt(px_new**2 + py_new**2)
        norm = np.maximum(norm / weight, 1)
        px = px_new / norm
        py = py_new / norm

        # Divergence
        div_x = np.diff(px, axis=0, prepend=np.zeros((1, u.shape[1])))
        div_y = np.diff(py, axis=1, prepend=np.zeros((u.shape[0], 1)))
        u = img + weight * (div_x + div_y)

    return u

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_test_image`, `generate_speckle_patterns`, `forward_operator`

In [ ]:
# ─── Data Generation ──────────────────────────────────────────────
def generate_test_image(size):
    """
    Create a test image with geometric features:
    - Letters/shapes for visual assessment
    - Smooth gradients for testing reconstruction fidelity
    """
    img = np.zeros((size, size))
    cx, cy = size // 2, size // 2

    # Circle
    Y, X = np.mgrid[:size, :size]
    r = np.sqrt((X - cx)**2 + (Y - cy)**2)
    img[r < size // 4] = 0.8

    # Square
    sq_s = size // 8
    img[cx-sq_s:cx+sq_s, size//4-sq_s:size//4+sq_s] = 0.6

    # Triangle
    for i in range(size // 6):
        j_start = 3 * size // 4 - i
        j_end = 3 * size // 4 + i
        row = cy - size // 6 + i
        if 0 <= row < size and 0 <= j_start and j_end < size:
            img[row, j_start:j_end] = 0.7

    # Small bright features
    img[size // 6, size // 6] = 1.0
    img[5 * size // 6, size // 6] = 1.0
    img[size // 6, 5 * size // 6] = 1.0
    img[5 * size // 6, 5 * size // 6] = 1.0

    # Smooth
    img = gaussian_filter(img, sigma=1)
    img = img / max(img.max(), 1e-12)

    return img

def generate_speckle_patterns(n_measurements, n_pixels, rng):
    """
    Generate random binary speckle illumination patterns.
    Models the spatial light modulator (SLM) patterns.
    """
    # Gaussian random patterns (better RIP properties)
    Phi = rng.standard_normal((n_measurements, n_pixels)) / np.sqrt(n_measurements)
    return Phi

def forward_operator(Phi, x, snr_db, rng):
    """
    Bucket detector measurement: b = Φ @ x + noise.
    """
    b_clean = Phi @ x
    sig_power = np.mean(b_clean**2)
    noise_power = sig_power / (10**(snr_db / 10))
    noise = np.sqrt(noise_power) * rng.standard_normal(len(b_clean))
    b_noisy = b_clean + noise
    return b_clean, b_noisy

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `ista_cs`, `fista_cs`

In [ ]:
def ista_cs(Phi, b, img_size, n_iter=300, lam=0.01):
    """
    ISTA (Iterative Shrinkage-Thresholding) for compressed sensing.
    Minimise: (1/2)||Φx - b||² + λ||Ψx||₁
    where Ψ is DCT sparsifying transform.
    """
    M, N = Phi.shape

    # Lipschitz constant
    L = np.linalg.norm(Phi.T @ Phi, ord=2)
    step = 1.0 / L

    x = np.zeros(N)
    print(f"  ISTA: {n_iter} iterations, λ={lam:.4f}, step={step:.6f}")

    for it in range(n_iter):
        # Gradient step
        residual = Phi @ x - b
        grad = Phi.T @ residual
        z = x - step * grad

        # Sparsity in DCT domain
        z_dct = dct2d_basis_transform(z, img_size)
        z_dct = soft_threshold(z_dct, lam * step)
        x = dct2d_basis_transform(z_dct, img_size, inverse=True)

        # Non-negativity
        x = np.maximum(x, 0)

        if (it + 1) % 50 == 0:
            obj = 0.5 * np.linalg.norm(residual)**2 + lam * np.sum(np.abs(z_dct))
            print(f"    iter {it+1:4d}: obj={obj:.4f}")

    return x

def fista_cs(Phi, b, img_size, n_iter=500, lam=0.01):
    """
    FISTA (Fast ISTA) with Nesterov acceleration.
    """
    M, N = Phi.shape
    L = np.linalg.norm(Phi.T @ Phi, ord=2)
    step = 1.0 / L

    x = np.zeros(N)
    y = x.copy()
    t = 1.0

    print(f"  FISTA: {n_iter} iterations, λ={lam:.4f}")

    for it in range(n_iter):
        # Gradient on y
        residual = Phi @ y - b
        grad = Phi.T @ residual
        z = y - step * grad

        # Proximal (DCT domain soft threshold)
        z_dct = dct2d_basis_transform(z, img_size)
        z_dct = soft_threshold(z_dct, lam * step)
        x_new = dct2d_basis_transform(z_dct, img_size, inverse=True)
        x_new = np.maximum(x_new, 0)

        # Nesterov momentum
        t_new = (1 + np.sqrt(1 + 4 * t**2)) / 2
        y = x_new + ((t - 1) / t_new) * (x_new - x)

        x = x_new
        t = t_new

        if (it + 1) % 100 == 0:
            obj = 0.5 * np.linalg.norm(Phi @ x - b)**2 + lam * np.sum(np.abs(z_dct))
            print(f"    iter {it+1:4d}: obj={obj:.4f}")

    return x

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ─── Metrics ───────────────────────────────────────────────────────
def compute_metrics(gt, rec):
    gt_n = gt / max(gt.max(), 1e-12)
    rec_n = rec / max(rec.max(), 1e-12)
    data_range = 1.0
    mse = np.mean((gt_n - rec_n)**2)
    psnr = float(10 * np.log10(data_range**2 / max(mse, 1e-30)))
    ssim_val = float(ssim_fn(gt_n, rec_n, data_range=data_range))
    cc = float(np.corrcoef(gt_n.ravel(), rec_n.ravel())[0, 1])
    re = float(np.linalg.norm(gt_n - rec_n) / max(np.linalg.norm(gt_n), 1e-12))
    rmse = float(np.sqrt(mse))
    return {"PSNR": psnr, "SSIM": ssim_val, "CC": cc, "RE": re, "RMSE": rmse}

# ─── Visualization ─────────────────────────────────────────────────
def visualize_results(gt, rec_corr, rec_fista, metrics, save_path):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title('Ground Truth')

    rec_corr_n = rec_corr / max(rec_corr.max(), 1e-12)
    axes[1].imshow(rec_corr_n, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Correlation GI')

    rec_fista_n = rec_fista / max(rec_fista.max(), 1e-12)
    axes[2].imshow(rec_fista_n, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title('FISTA CS')

    err = np.abs(gt - rec_fista_n)
    axes[3].imshow(err, cmap='hot', vmin=0)
    axes[3].set_title('|Error|')

    for ax in axes:
        ax.axis('off')

    fig.suptitle(
        f"Ghost Imaging — Compressive Single-Pixel Reconstruction\n"
        f"M/N={COMPRESSION_RATIO:.0%} | PSNR={metrics['PSNR']:.1f} dB | "
        f"SSIM={metrics['SSIM']:.4f} | CC={metrics['CC']:.4f}",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.9])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 70)
print("  Ghost Imaging — Compressive Single-Pixel Reconstruction")
print("=" * 70)

rng = np.random.default_rng(SEED)

### Stage 1: Data Generation

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Stage 1: Data Generation
print("\n[STAGE 1] Data Generation")
img_gt = generate_test_image(IMG_SIZE)
x_gt = img_gt.ravel()
print(f"  Image: {IMG_SIZE}×{IMG_SIZE} = {N_PIXELS} pixels")
print(f"  Measurements: {N_MEASUREMENTS} (ratio={COMPRESSION_RATIO:.0%})")

### Stage 2: Forward — Measurement

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Stage 2: Forward — Measurement
print("\n[STAGE 2] Forward — Speckle Illumination + Bucket Detection")
Phi = generate_speckle_patterns(N_MEASUREMENTS, N_PIXELS, rng)
b_clean, b_noisy = forward_operator(Phi, x_gt, NOISE_SNR_DB, rng)
print(f"  Measurement matrix Φ: {Phi.shape}")
print(f"  Bucket signal range: [{b_noisy.min():.3f}, {b_noisy.max():.3f}]")

### Stage 3a: Correlation GI

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Stage 3a: Correlation GI
print("\n[STAGE 3a] Inverse — Correlation Ghost Imaging")
x_corr = correlation_gi(Phi, b_noisy)
x_corr = np.maximum(x_corr, 0)
img_corr = x_corr.reshape(IMG_SIZE, IMG_SIZE)
m_corr = compute_metrics(img_gt, img_corr)
print(f"  Correlation GI: CC={m_corr['CC']:.4f}, PSNR={m_corr['PSNR']:.1f}")

### Stage 3b: FISTA CS

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Stage 3b: FISTA CS
print("\n[STAGE 3b] Inverse — FISTA Compressed Sensing")
x_fista = fista_cs(Phi, b_noisy, IMG_SIZE, n_iter=1000, lam=0.001)
img_fista = x_fista.reshape(IMG_SIZE, IMG_SIZE)
img_fista = np.clip(img_fista, 0, 1)

# TV post-processing
img_fista = tv_denoise(img_fista, weight=0.03, n_iter=50)
m_fista = compute_metrics(img_gt, img_fista)
print(f"  FISTA: CC={m_fista['CC']:.4f}, PSNR={m_fista['PSNR']:.1f}")

# Choose best
if m_fista['CC'] >= m_corr['CC']:
    img_rec = img_fista
    metrics = m_fista
    method = "FISTA"
else:
    img_rec = img_corr
    metrics = m_corr
    method = "Correlation"
print(f"\n  → Using {method} (higher CC)")

### Stage 4: Evaluation

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Stage 4: Evaluation
print("\n[STAGE 4] Evaluation Metrics:")
for k, v in sorted(metrics.items()):
    print(f"  {k:20s} = {v}")

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), img_rec)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), img_gt)

visualize_results(img_gt, img_corr, img_fista, metrics,
                  os.path.join(RESULTS_DIR, "reconstruction_result.png"))

print("\n" + "=" * 70)
print("  DONE — Results saved to", RESULTS_DIR)
print("=" * 70)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **ghost_imaging**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=50.18 dB ← 🔧 修复前: 14.11 dB, SSIM=0.998)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: See documentation
- Repository: https://github.com/jovenitti/GhostImaging
